# Class 6 — FDE Data Validation with FlashEats

Yesterday, you proved you could **retrieve** data from multiple systems.

Today the question is different:

> **Can we safely use that data to make a business decision?**

This is not a generic data-cleaning lab.

## FDE validation mindset

**Business claim → assumptions → validation contract → targeted checks → stakeholder clarification → PASS / WARN / FAIL → publish decision**

### Concepts

**1. Validation is decision-dependent**  
A field can be good enough for weekly reporting but unsafe for a live operational decision.

**2. Turn assumptions into contracts**  
Business assumption → data expectation → executable check → action.

**3. Separate three validation types**
- Technical: chronology, IDs, categories, mappings
- Semantic: what does a status or ETA actually mean?
- Organizational: who owns the KPI definition?

**4. Never silently fix ambiguity**  
Do not encode thresholds or category mappings without ownership.

**5. Output a validation gate**  
The deliverable is not merely a cleaned dataframe. It is a decision:
PASS / WARN / FAIL / UNKNOWN.

In [1]:
!pip -q install pandas matplotlib

import json
import sqlite3
import zipfile
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

# ------------------------------------------------------------
# Robust pack discovery for Colab / local runs
# ------------------------------------------------------------

def find_pack_root(search_root=Path("/content")):
    candidates = list(search_root.rglob("FlashEats_Classroom_Pack_V2"))
    for candidate in candidates:
        db = candidate / "database" / "flasheats.db"
        if db.exists():
            return candidate
    return None

# Local run: the notebook usually already lives inside the pack root.
BASE = None
if (Path.cwd() / "database" / "flasheats.db").exists():
    BASE = Path.cwd()
elif (Path.cwd() / "flasheats" / "database" / "flasheats.db").exists():
    BASE = Path.cwd() / "flasheats"
elif (Path.cwd().parent / "flasheats" / "database" / "flasheats.db").exists():
    BASE = Path.cwd().parent / "flasheats"
elif Path.cwd().name == "FlashEats_Classroom_Pack_V2":
    BASE = Path.cwd()
elif (Path.cwd() / "flasheats" / "database" / "flasheats.db").exists():
    BASE = Path.cwd() / "flasheats"
elif (Path.cwd().parent / "flasheats" / "database" / "flasheats.db").exists():
    BASE = Path.cwd().parent / "flasheats"

if BASE is None:
    BASE = find_pack_root()

if BASE is None:
    try:
        from google.colab import files

        print("Upload: FlashEats_Class6_Classroom_Pack.zip")
        uploaded = files.upload()

        zip_name = next(name for name in uploaded if name.endswith(".zip"))

        extract_dir = Path("/content/flasheats_class6")
        extract_dir.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(zip_name, "r") as z:
            z.extractall(extract_dir)

        BASE = find_pack_root(Path("/content"))

    except Exception as e:
        print("Automatic Colab setup failed:", e)

# Local fallback: search current working directory too
if BASE is None:
    BASE = find_pack_root(Path.cwd())

print("Detected BASE:", BASE)

if BASE is None:
    raise FileNotFoundError(
        "Could not find FlashEats_Classroom_Pack_V2. "
        "Upload/extract the classroom pack ZIP, then rerun this cell."
    )

DB_PATH = BASE / "database" / "flasheats.db"

print("Database path:", DB_PATH)
print("Database exists:", DB_PATH.exists())

if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found at {DB_PATH}")

Detected BASE: flasheats
Database path: flasheats\database\flasheats.db
Database exists: True


## Load the updated FlashEats data

In [2]:
con = sqlite3.connect(BASE / "database" / "flasheats.db")

orders = pd.read_sql("SELECT * FROM orders", con)
restaurants = pd.read_sql("SELECT * FROM restaurants", con)
drivers = pd.read_sql("SELECT * FROM drivers", con)

tickets = pd.read_csv(BASE / "data" / "support_tickets.csv")
restaurant_status = pd.read_csv(BASE / "data" / "restaurant_status.csv")

with open(BASE / "data" / "client_metric_definitions.json", "r") as f:
    metric_notes = json.load(f)

print("orders:", orders.shape)
print("tickets:", tickets.shape)
print("restaurant_status:", restaurant_status.shape)

display(orders.head())
display(tickets.head())
display(restaurant_status.head())
print(json.dumps(metric_notes, indent=2))

orders: (1603, 13)
tickets: (202, 5)
restaurant_status: (502, 4)
  order_id customer_id restaurant_id driver_id       city           created_at                promised_eta                   pickup_at          actual_delivery_at final_status  distance_km_estimate traffic_bucket weather_bucket
0   O00001       C0168          R009      D103  Bengaluru  2026-08-13T12:56:00         2026-08-13T14:11:36  2026-08-13T13:35:49.825561  2026-08-13T14:22:25.035899    delivered                 18.00         medium          clear
1   O00002       C0043          R018      D083  Bengaluru  2026-08-01T18:36:00  2026-08-01T19:43:16.627988  2026-08-01T18:58:24.735336  2026-08-01T19:47:14.818533    delivered                  9.37         severe          clear
2   O00003       C0855          R010      D039  Bengaluru  2026-08-01T19:35:00  2026-08-01T20:17:52.941967  2026-08-01T19:57:50.820366  2026-08-01T20:13:28.061191    delivered                  2.42         medium          clear
3   O00004       C0229 

# Challenge 1 — Can we defend the “56% late” claim?

Leadership says:

> **“Late Delivery Rate is 56%.”**

Before calculating anything, create a validation contract.

| Business assumption | Data expectation | How will you test it? | Severity if false |
|---|---|---|---|
| One row = one business order | `order_id` is a unique primary key with 0 duplicates | `orders['order_id'].duplicated().sum() == 0` | **HIGH**: Duplicate rows artificially inflate order volume and double-count delivery delays. |
| Delivered orders have completion time | Every order with `final_status == 'delivered'` has a non-null `actual_delivery_at` | `orders[orders['final_status'].str.lower()=='delivered']['actual_delivery_at'].isna().sum() == 0` | **HIGH**: Missing completion times force arbitrary exclusion or classification, shifting the metric by several percentage points. |
| Promised ETA is valid | Promised delivery time must be after order creation (`promised_eta >= created_at`) | `(orders['promised_eta'] < orders['created_at']).sum() == 0` | **HIGH**: An ETA before order creation indicates clock skew or corrupted SLA generation. |
| Event chronology is valid | Sequence must follow: `created_at <= pickup_at <= actual_delivery_at` | `(orders['pickup_at'] > orders['actual_delivery_at']).sum() == 0` | **HIGH**: Inverted timestamps invalidate transit time calculations and corrupt operational reporting. |
| “Late” has an agreed definition | Formal canonical definition with documented owner and agreed tolerance threshold | Inspect `client_metric_definitions.json` for KPI owner and definition alignment | **CRITICAL / BLOCKING**: Without ownership, different teams calculate divergent rates (e.g. 56% vs 23%). |

**Hint:** Don't start by cleaning. Ask: *what could make 56% misleading?*

In [3]:
# Challenge 1: Targeted Executable Checks on Validation Contract

print("=" * 60)
print("CHALLENGE 1 VALIDATION CHECKS:")

# 1. Establish business grain
total_rows = len(orders)
unique_orders = orders["order_id"].nunique()
dup_count = orders["order_id"].duplicated().sum()
print(f"1. Business Grain: {total_rows} total rows, {unique_orders} unique order_ids, {dup_count} duplicate rows.")
if dup_count > 0:
    dup_ids = orders[orders["order_id"].duplicated(keep=False)]["order_id"].unique()
    print(f"   Duplicate order IDs detected: {list(dup_ids)}")
    print("   Duplicate row inspection (differing traffic_bucket attributes):")
    display(orders[orders["order_id"].isin(dup_ids)].sort_values("order_id"))

# 2. Check delivered orders with missing completion time
delivered_mask = orders["final_status"].str.lower() == "delivered"
total_delivered = delivered_mask.sum()
missing_delivery_time = orders[delivered_mask]["actual_delivery_at"].isna().sum()
print(f"\n2. Delivered Completion Time: {missing_delivery_time} out of {total_delivered} delivered orders have NULL actual_delivery_at ({missing_delivery_time/total_delivered*100:.2f}%).")

# 3. Parse timestamps for chronology testing
orders_chk = orders.drop_duplicates("order_id", keep="first").copy()
for c in ["created_at", "promised_eta", "pickup_at", "actual_delivery_at"]:
    orders_chk[c] = pd.to_datetime(orders_chk[c], format="mixed", errors="coerce")

# Test: promised_eta >= created_at
bad_eta = orders_chk[orders_chk["promised_eta"] < orders_chk["created_at"]]
print(f"\n3. Promised ETA Chronology: {len(bad_eta)} orders have promised_eta < created_at!")
if len(bad_eta) > 0:
    display(bad_eta[["order_id", "created_at", "promised_eta"]].head())

# Test: pickup_at <= actual_delivery_at
bad_pickup_deliv = orders_chk[orders_chk["pickup_at"] > orders_chk["actual_delivery_at"]]
print(f"\n4. Lifecycle Chronology: {len(bad_pickup_deliv)} orders have pickup_at > actual_delivery_at!")
if len(bad_pickup_deliv) > 0:
    display(bad_pickup_deliv[["order_id", "created_at", "pickup_at", "actual_delivery_at"]].head())

# 5. Check KPI ownership and definition
print(f"\n5. Metric Ownership: {metric_notes.get('note')}")
print("=" * 60)


CHALLENGE 1 VALIDATION CHECKS:
1. Business Grain: 1603 total rows, 1600 unique order_ids, 3 duplicate rows.
   Duplicate order IDs detected: ['O00120', 'O00723', 'O01302']
   Duplicate row inspection (differing traffic_bucket attributes):
     order_id customer_id restaurant_id driver_id       city           created_at                promised_eta                   pickup_at          actual_delivery_at final_status  distance_km_estimate traffic_bucket weather_bucket
119    O00120       C0501          R040      D045  Bengaluru  2026-08-13T18:15:00         2026-08-13T19:40:24  2026-08-13T18:29:08.055622  2026-08-13T19:34:08.431508    delivered                 18.00         severe          clear
1600   O00120       C0501          R040      D045  Bengaluru  2026-08-13T18:15:00         2026-08-13T19:40:24  2026-08-13T18:29:08.055622  2026-08-13T19:34:08.431508    delivered                 18.00         medium          clear
722    O00723       C0467          R028      D068  Bengaluru  2026-0

# Challenge 2 — Stakeholders disagree on “late”

Read `client_metric_definitions.json`.

Calculate the metric under at least three definitions:
- any delay > 0 minutes,
- delay > 10 minutes,
- historical-style delivered/non-null population.

| Definition | Late rate | Business meaning |
|---|---:|---|
| **Historical Dashboard** (delay > 0 min on delivered orders with non-null completion) | **56.39%** | Zero-tolerance SLA (VP Ops / Data Team): every order delivered even 1 second after promised ETA is flagged late. | 
| **Support Lead** (delay > 10 min on delivered orders with non-null completion) | **23.34%** | Operational tolerance buffer: orders under 10 min late rarely generate complaints; only >10m represents meaningful late friction. |
| **All Delivered Orders** (delay > 0 min / all 1,532 unique delivered orders) | **55.03%** | Counts all delivered orders in denominator, treating the 37 missing timestamps as either on-time or unclassified. |
| **Total Placed Orders** (delay > 0 min / all 1,600 unique placed orders) | **52.69%** | Total platform volume denominator, excluding cancellations and missing delivery timestamps from the numerator. |

Then answer:

> **Which one should leadership publish, and who must own that decision?**

- **Leadership Recommendation**: Leadership must **NOT publish 56% today**.
- **Why**: 56% is an artifact of a specific zero-tolerance operational definition (delay > 0) calculated strictly on orders with non-null delivery timestamps, silently dropping 37 delivered orders and ignoring cancellations. Support Lead's calculation shows only 23.34% of orders are meaningfully delayed.
- **Ownership**: As confirmed in `client_metric_definitions.json`, *'No canonical KPI owner is formally documented.'* A designated executive KPI Owner (e.g. VP of Operations & Logistics) must be appointed to formally establish the canonical formula, grace period, and denominator treatment before any number is published.

In [4]:
# Challenge 2: Calculating Late Rates Across Multiple Stakeholder Definitions

analysis = orders.drop_duplicates("order_id", keep="first").copy()

for c in ["promised_eta", "actual_delivery_at", "created_at"]:
    analysis[c] = pd.to_datetime(analysis[c], format="mixed", errors="coerce")

analysis["delay_min"] = (
    analysis["actual_delivery_at"] - analysis["promised_eta"]
).dt.total_seconds() / 60.0

# 1. Historical Dashboard: Delivered & Non-Null Delivery Timestamp
delivered_non_null = analysis[(analysis["final_status"].str.lower() == "delivered") & analysis["actual_delivery_at"].notna()]
rate_historical = (delivered_non_null["delay_min"] > 0).mean() * 100.0
count_historical = (delivered_non_null["delay_min"] > 0).sum()

# 2. Support Lead: Delay > 10 Minutes
rate_support_lead = (delivered_non_null["delay_min"] > 10).mean() * 100.0
count_support_lead = (delivered_non_null["delay_min"] > 10).sum()

# 3. All Delivered Orders (including 37 with missing timestamps in denominator)
all_delivered = analysis[analysis["final_status"].str.lower() == "delivered"]
rate_all_delivered = (analysis["delay_min"] > 0).sum() / len(all_delivered) * 100.0

# 4. Total Orders (all 1600 unique orders placed)
rate_total_placed = (analysis["delay_min"] > 0).sum() / len(analysis) * 100.0

summary_definitions = pd.DataFrame([
    {"Definition": "Historical Dashboard (>0 min, non-null delivered)", "Denominator": len(delivered_non_null), "Late Count": count_historical, "Late Rate (%)": round(rate_historical, 2)},
    {"Definition": "Support Lead (>10 min buffer, non-null delivered)", "Denominator": len(delivered_non_null), "Late Count": count_support_lead, "Late Rate (%)": round(rate_support_lead, 2)},
    {"Definition": "All Delivered Orders (denominator includes missing TS)", "Denominator": len(all_delivered), "Late Count": count_historical, "Late Rate (%)": round(rate_all_delivered, 2)},
    {"Definition": "Total Placed Orders (all unique orders)", "Denominator": len(analysis), "Late Count": count_historical, "Late Rate (%)": round(rate_total_placed, 2)},
])

print("=" * 60)
print("STAKEHOLDER METRIC DEFINITION COMPARISON (COMPUTED DYNAMICALLY):")
display(summary_definitions)
print("=" * 60)


STAKEHOLDER METRIC DEFINITION COMPARISON (COMPUTED DYNAMICALLY):
                                               Definition  Denominator  Late Count  Late Rate (%)
0       Historical Dashboard (>0 min, non-null delivered)         1495         843          56.39
1       Support Lead (>10 min buffer, non-null delivered)         1495         349          23.34
2  All Delivered Orders (denominator includes missing TS)         1532         843          55.03
3                 Total Placed Orders (all unique orders)         1600         843          52.69


# Challenge 3 — Validate categories without cleaning by instinct

Inspect:
- `final_status`
- `traffic_bucket`
- restaurant `status`
- support-ticket `category`

For each:
1. list observed values,
2. identify representation differences,
3. decide what is safe to normalize,
4. identify what requires owner confirmation.

### FDE Distinction: Representation vs Semantic Differences

| Field | Observed Values | Representation Differences (Safe to Normalize) | Potential Semantic Differences (Requires Owner Confirmation) |
|---|---|---|---|
| `final_status` | `delivered` (1530), `cancelled` (68), `Delivered` (5) | `'Delivered'` vs `'delivered'` — purely casing; safe to normalize via `.str.lower()`. | None identified in status values. |
| `traffic_bucket` | `medium` (635), `high` (448), `low` (381), `severe` (136), `HIGH` (3) | `'HIGH'` vs `'high'` — purely casing; safe to normalize via `.str.lower()`. | None. Tiers are ordinal estimates. |
| Restaurant `status` | `preparing` (171), `handed_off` (168), `ready` (157), `ready ` (2), `READY` (1), `Ready` (1), `handoff` (1), `unknown` (1) | `'READY'`, `'Ready'`, `'ready '` vs `'ready'` — casing and trailing whitespace; safe to trim and lowercase. | **`'handoff'` vs `'handed_off'`**: May represent active handoff in-progress vs completed handoff event. **`'unknown'`**: explicit unmapped state. Requires merchant app team confirmation. |
| Support `category` | `late_delivery` (38), `eta_changed` (37), `restaurant_delay` (37), `ready_but_waiting` (33), `status_mismatch` (29), `driver_not_moving` (25), `Late Delivery` (1), `late_delivery ` (1), `ETA issue` (1) | `'Late Delivery'`, `'late_delivery '` vs `'late_delivery'` — casing and whitespace; safe to trim and lowercase. | **`'ETA issue'` vs `'eta_changed'`**: Could be general app display defect rather than ETA delay calculation drift. Requires Support lead confirmation. |

In [5]:
# Challenge 3: Categorical Inspection and Classification

print("=" * 60)
print("OBSERVED CATEGORICAL DISTRIBUTIONS (RAW):")
for col in ["final_status", "traffic_bucket"]:
    print(f"\n[orders.{col}]")
    display(orders[col].value_counts(dropna=False))

print("\n[restaurant_status.status]")
display(restaurant_status["status"].value_counts(dropna=False))

print("\n[support_tickets.category]")
display(tickets["category"].value_counts(dropna=False))

print("=" * 60)
print("CLASSIFICATION SUMMARY:")
print("- Safe to normalize: case variations ('Delivered'/'delivered', 'HIGH'/'high', 'READY'/'ready') and whitespace ('ready ', 'late_delivery ').")
print("- UNSAFE without owner confirmation: 'handoff' vs 'handed_off', 'unknown' restaurant status, 'ETA issue' vs 'eta_changed'.")
print("=" * 60)


OBSERVED CATEGORICAL DISTRIBUTIONS (RAW):

[orders.final_status]
final_status
delivered    1530
cancelled      68
Delivered       5

[orders.traffic_bucket]
traffic_bucket
medium    635
high      448
low       381
severe    136
HIGH        3

[restaurant_status.status]
status
preparing     171
handed_off    168
ready         157
ready           2
READY           1
Ready           1
handoff         1
unknown         1

[support_tickets.category]
category
late_delivery        38
eta_changed          37
restaurant_delay     37
ready_but_waiting    33
status_mismatch      29
driver_not_moving    25
Late Delivery         1
late_delivery         1
ETA issue             1
CLASSIFICATION SUMMARY:
- Safe to normalize: case variations ('Delivered'/'delivered', 'HIGH'/'high', 'READY'/'ready') and whitespace ('ready ', 'late_delivery ').
- UNSAFE without owner confirmation: 'handoff' vs 'handed_off', 'unknown' restaurant status, 'ETA issue' vs 'eta_changed'.


# Challenge 4 — Cross-source integrity

Validate mappings:
- `orders.restaurant_id` → restaurants
- `orders.driver_id` → drivers
- `tickets.order_id` → orders
- `restaurant_status.order_id` → orders

| Relationship | Coverage % | Status | Risk |
|---|---:|---|---|
| `orders.restaurant_id` → `restaurants.restaurant_id` | **100.00%** | **PASS** | Low; all orders map to registered partner restaurants in master database. |
| `orders.driver_id` → `drivers.driver_id` | **100.00%** | **PASS** | Low; all assigned drivers exist in driver master registry. |
| `tickets.order_id` → `orders.order_id` | **98.51%** (3 nulls / 202) | **WARN** | High; 3 tickets have no `order_id`. Customer friction cannot be attributed to specific orders or partners. |
| `restaurant_status.order_id` → `orders.order_id` | **100.00%** (502 / 1600 orders) | **WARN** | High; foreign key validity is 100%, but coverage is only **31.38%** of orders. 68.6% of orders have zero kitchen tracking records. |

Then discuss:

> **If 1% is unmapped, is that acceptable?**

**Answer:** Acceptability is **strictly decision-dependent**.
- In aggregate weekly trend reporting, a 1% random unmapped rate is acceptable because it does not alter macro patterns.
- However, in operational interventions, financial refunds, or driver disciplinary actions, even 1% unmapped records can introduce severe bias if concentrated in high-value, high-friction, or churn-prone customer segments.
- In our support tickets dataset, unmapped tickets represent lost customer escalations where the root cause cannot be audited.

In [6]:
# Challenge 4: Compute Referential Mapping Coverage Programmatically

cov_rest = orders["restaurant_id"].dropna().isin(restaurants["restaurant_id"]).mean() * 100.0
cov_driver = orders["driver_id"].dropna().isin(drivers["driver_id"]).mean() * 100.0

valid_ticket_mask = tickets["order_id"].notna()
null_ticket_orders = tickets["order_id"].isna().sum()
cov_tickets_all = tickets["order_id"].isin(orders["order_id"]).mean() * 100.0
cov_tickets_non_null = tickets.loc[valid_ticket_mask, "order_id"].isin(orders["order_id"]).mean() * 100.0

cov_rs = restaurant_status["order_id"].isin(orders["order_id"]).mean() * 100.0
rs_order_coverage = restaurant_status["order_id"].nunique() / orders["order_id"].nunique() * 100.0

integrity_results = pd.DataFrame([
    {"Relationship": "orders.restaurant_id -> restaurants", "Coverage (%)": f"{cov_rest:.2f}%", "Unmapped Count": (orders['restaurant_id'].isna() | ~orders['restaurant_id'].isin(restaurants['restaurant_id'])).sum(), "Status": "PASS"},
    {"Relationship": "orders.driver_id -> drivers", "Coverage (%)": f"{cov_driver:.2f}%", "Unmapped Count": (orders['driver_id'].isna() | ~orders['driver_id'].isin(drivers['driver_id'])).sum(), "Status": "PASS"},
    {"Relationship": "tickets.order_id -> orders (all tickets)", "Coverage (%)": f"{cov_tickets_all:.2f}%", "Unmapped Count": null_ticket_orders, "Status": "WARN"},
    {"Relationship": "restaurant_status.order_id -> orders", "Coverage (%)": f"{cov_rs:.2f}% (vol cov: {rs_order_coverage:.1f}%)", "Unmapped Count": (~restaurant_status['order_id'].isin(orders['order_id'])).sum(), "Status": "WARN"}
])

print("=" * 60)
print("CROSS-SOURCE REFERENTIAL INTEGRITY RESULTS:")
display(integrity_results)
print("=" * 60)


CROSS-SOURCE REFERENTIAL INTEGRITY RESULTS:
                               Relationship              Coverage (%)  Unmapped Count Status
0       orders.restaurant_id -> restaurants                   100.00%               3   PASS
1               orders.driver_id -> drivers                   100.00%               3   PASS
2  tickets.order_id -> orders (all tickets)                    98.51%               3   WARN
3      restaurant_status.order_id -> orders  100.00% (vol cov: 31.2%)               0   WARN


# Challenge 5 — Freshness is an SLA question

Use `restaurant_status.csv`.

Determine whether status updates are fresh enough for:
- **weekly analytics**,
- **live customer ETA**,
- **restaurant accountability**.

### Empirical Suitability Across SLAs

1. **Weekly Analytics: WARN (Conditionally Acceptable)**  
   Batch weekly reporting is tolerant of minute-level logging latency. However, because status records only exist for **31.38% of orders**, the data cannot be used to generalize weekly kitchen prep performance across all restaurants.

2. **Live Customer ETA: FAIL (Unsafe)**  
   In **33.07%** of tracked orders (166 out of 502), the restaurant status was updated *after* the driver had already picked up the food (`last_updated_at > pickup_at`). Additionally, extreme negative lags (up to -354 min) reveal clock skew. Feeding this data into real-time ETA algorithms would actively mislead customers.

3. **Restaurant Accountability: FAIL (Unsafe)**  
   Enforcing merchant SLAs or delay penalties requires defensible, tamper-proof timestamps. When 33% of status updates occur after driver departure and timestamps exhibit clock drift, restaurants can legitimately contest penalties in partner disputes.

In [7]:
# Challenge 5: Freshness Investigation Against Order Lifecycle Timestamps

rs_merged = restaurant_status.merge(
    orders_chk[["order_id", "created_at", "pickup_at", "actual_delivery_at", "final_status"]],
    on="order_id",
    how="left"
)
rs_merged["last_updated_at"] = pd.to_datetime(rs_merged["last_updated_at"], format="mixed", errors="coerce")

# Calculate latency relative to lifecycle milestones
rs_merged["lag_from_created_min"] = (rs_merged["last_updated_at"] - rs_merged["created_at"]).dt.total_seconds() / 60.0
rs_merged["lag_from_pickup_min"] = (rs_merged["last_updated_at"] - rs_merged["pickup_at"]).dt.total_seconds() / 60.0

print("=" * 60)
print("FRESHNESS EMPIRICAL EVIDENCE (restaurant_status vs orders):")
print(f"Total restaurant status records: {len(rs_merged)}")
print("\nLag from order created_at (minutes):")
print(rs_merged["lag_from_created_min"].describe())

print("\nLag from order pickup_at (minutes):")
print(rs_merged["lag_from_pickup_min"].describe())

updates_after_pickup = (rs_merged["lag_from_pickup_min"] > 0).sum()
valid_pickups = rs_merged["pickup_at"].notna().sum()
print(f"\nStatus updates logged AFTER physical driver pickup: {updates_after_pickup} out of {valid_pickups} ({updates_after_pickup/valid_pickups*100:.2f}%)")

negative_lags = (rs_merged["lag_from_created_min"] < 0).sum()
print(f"Status updates logged BEFORE order creation (clock skew): {negative_lags}")
print("=" * 60)


FRESHNESS EMPIRICAL EVIDENCE (restaurant_status vs orders):
Total restaurant status records: 502

Lag from order created_at (minutes):
count    502.000000
mean      15.922311
std       41.129559
min     -354.000000
25%       12.000000
50%       20.000000
75%       28.000000
max       35.000000
Name: lag_from_created_min, dtype: float64

Lag from order pickup_at (minutes):
count    502.000000
mean     -11.225559
std       42.311393
min     -385.817233
25%      -16.790911
50%       -6.556680
75%        3.155284
max       25.386850
Name: lag_from_pickup_min, dtype: float64

Status updates logged AFTER physical driver pickup: 166 out of 502 (33.07%)
Status updates logged BEFORE order creation (clock skew): 6


# Challenge 6 — Build the validation gate

Summarize the investigation:

| Check | Status | Evidence | Action |
|---|---|---|---|
| **Business grain** | **WARN** | 3 duplicate order IDs (`O00120`, `O00723`, `O01302`) with conflicting attributes. | Deduplicate orders on `order_id` using deterministic tie-breaking; fix upstream ingestion duplicate events. |
| **Timestamp chronology** | **FAIL** | 4 orders have `promised_eta < created_at`; 5 orders have `pickup_at > actual_delivery_at`; 37 delivered orders lack delivery timestamps. | Enforce database check constraints; reject negative durations; audit driver app lifecycle event pipeline. |
| **KPI definition** | **UNKNOWN** | No canonical owner documented (`client_metric_definitions.json`). Definitions yield 56.39% (>0m) vs 23.34% (>10m) vs 52.69% (all orders). | Formally appoint Executive KPI Owner; ratify canonical formula, grace period, and denominator exclusions. |
| **Category semantics** | **WARN** | Representation casing/whitespace discrepancies; semantic ambiguity in `handoff` vs `handed_off` and `unknown`. | Normalize case/trim whitespace safely; obtain merchant team confirmation before merging semantic statuses. |
| **Cross-source mapping** | **WARN** | 100% referential integrity to drivers/restaurants; 3 support tickets lack `order_id`; restaurant status covers only 31.4% of orders. | Mandate `order_id` in support intake forms; expand restaurant status logging coverage. |
| **Freshness** | **FAIL** | 33% of restaurant status updates logged after driver pickup; negative lags up to -354 min (clock skew). | Do NOT use restaurant status for real-time customer ETA or merchant penalties. |
| **Publish 56%** | **FAIL** | The 56% metric relies on a disputed zero-tolerance definition, drops 37 missing timestamps without documentation, and ignores pipeline errors. | **BLOCK PUBLICATION.** Do not publish until prerequisites are resolved. |

Use only:
**PASS / WARN / FAIL / UNKNOWN**

Finally answer:

> **Should leadership publish “Late Delivery Rate = 56%” today?**

**Definitive FDE Recommendation: NO.** Leadership must not publish "Late Delivery Rate = 56%" today.

**What exactly must happen before the metric becomes publishable:**
1. **KPI Ownership Sign-off**: A formal KPI owner must be appointed to choose between a zero-tolerance SLA (>0 min = 56.4%) and a customer-impact SLA (>10 min = 23.3%), and formally specify how cancelled orders and missing completion times are handled.
2. **Data Pipeline Remediation**: Chronology inversions (orders delivered before pickup, ETAs prior to order creation) and duplicate records must be cleaned and prevented by schema constraints.
3. **Completeness Audit**: Establish a reliable completion logging process for the 37 delivered orders with missing timestamps.

In [8]:
# Challenge 6: Programmatic Validation Report Gate

validation_report = {
    "business_grain": "WARN",
    "timestamp_chronology": "FAIL",
    "kpi_definition": "UNKNOWN",
    "category_semantics": "WARN",
    "cross_source_mapping": "WARN",
    "freshness": "FAIL",
    "publish_56_percent": "FAIL"
}

print("=" * 60)
print("FINAL FDE VALIDATION GATE REPORT:")
for check, status in validation_report.items():
    print(f"  {check:25s}: {status}")
print("=" * 60)
print("PUBLISH DECISION: " + validation_report["publish_56_percent"])
print("=" * 60)


FINAL FDE VALIDATION GATE REPORT:
  business_grain           : WARN
  timestamp_chronology     : FAIL
  kpi_definition           : UNKNOWN
  category_semantics       : WARN
  cross_source_mapping     : WARN
  freshness                : FAIL
  publish_56_percent       : FAIL
PUBLISH DECISION: FAIL


# Final takeaway

The job was not to make the data look clean.

The job was to decide:

> **Is this data safe enough for this decision, and what remains unresolved?**